In [128]:
!pip install -q torch torchvision onnx onnxruntime onnxruntime-tools onnxscript

In [129]:
import torch
import torch.nn as nn
import onnx

from torchvision.models import mobilenet_v3_small
from onnxruntime.quantization import quantize_dynamic, QuantType

In [130]:
from torchvision import models

num_classes = 9

# Create the model with the correct number of classes
model = models.mobilenet_v3_small(
    weights=None,
    num_classes=num_classes
)

# Load the checkpoint
state_dict = torch.load(
    "best_mobilenetv3_small.pth",
    map_location="cpu"
)

# Handle checkpoints that wrap the state dict
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]

model.load_state_dict(state_dict)

model.eval()

print("✓ Model loaded successfully")

✓ Model loaded successfully


In [131]:
dummy_input = torch.randn(1, 3, 224, 224)

In [132]:
torch.onnx.export(
    model,
    dummy_input,
    "best_mobilenetv3_small_fp32.onnx",
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamo=False,          # <-- important
)

print("✓ FP32 ONNX exported")
import onnx

model = onnx.load("best_mobilenetv3_small_fp32.onnx")

print("IR version:", model.ir_version)
print("Producer:", model.producer_name)
print("Producer version:", model.producer_version)

for opset in model.opset_import:
    print("Domain:", opset.domain)
    print("Opset:", opset.version)

onnx.checker.check_model(model)
print("Model is valid")


/tmp/ipykernel_1090/4263701229.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✓ FP32 ONNX exported
IR version: 7
Producer: pytorch
Producer version: 2.11.0
Domain: 
Opset: 13
Model is valid


In [140]:
import onnxruntime as ort

session = ort.InferenceSession("best_mobilenetv3_small_fp32.onnx")

print("Inputs:")
for i in session.get_inputs():
    print(i.name, i.shape, i.type)

print("\nOutputs:")
for o in session.get_outputs():
    print(o.name, o.shape, o.type)
print(sorted(set(node.op_type for node in model.graph.node)))

Inputs:
input [1, 3, 224, 224] tensor(float)

Outputs:
output [1, 9] tensor(float)
['Add', 'Conv', 'Flatten', 'Gemm', 'GlobalAveragePool', 'HardSigmoid', 'Mul', 'Relu']


In [133]:
onnx_model = onnx.load("best_mobilenetv3_small_fp32.onnx")

onnx.checker.check_model(onnx_model)

print("✓ ONNX model is valid")

✓ ONNX model is valid


In [134]:
import torch
import onnx
import onnxruntime

print(torch.__version__)
print(onnx.__version__)
print(onnxruntime.__version__)

2.11.0+cpu
1.22.0
1.27.0
